# Single PHOEBE + SPICE eclipse system

Builds **one** binary the same way `check_phoebe_eclipses.py` does, runs both PHOEBE and SPICE on the eclipse window, and plots the bolometric light curves side by side.

Useful for interactive exploration / plot tweaks without launching the full grid.

In [2]:
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('XLA_PYTHON_CLIENT_ALLOCATOR', 'platform')

import jax
jax.config.update('jax_enable_x64', True)
jax.config.update('jax_platform_name', 'cpu')

import sys
sys.path.append('/Users/mjablons/code/spice/src')

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from astropy import units as u

import phoebe
from phoebe.parameters.dataset import _mesh_columns

from spice.models.binary import Binary, add_orbit, evaluate_orbit_at_times
from spice.models.mesh_model import IcosphereModel
from spice.models.mesh_view import get_mesh_view
from spice.models.orbit_utils import eclipse_timestamps_kepler
from spice.spectrum.blackbody import Blackbody
from spice.spectrum.filter import Bolometric
from spice.spectrum.spectrum import AB_passband_luminosity, simulate_observed_flux

%matplotlib inline

DAYS_TO_YR = 0.0027378507871321013
DEG_TO_RAD = 0.017453292519943295

## Parameters

Edit these to explore other geometries. Defaults match a clean, fully eclipsing case.

In [3]:
inclination     = 90.0   # degrees
period          = 5.0    # days
q               = 1.0    # mass ratio
ecc             = 0.0    # eccentricity
primary_mass    = 1.0    # Msun
n_times         = 40     # samples (split evenly between primary/secondary eclipse)
n_mesh_elements = 1000   # both SPICE icosphere and PHOEBE ntriangles

## Build PHOEBE bundle and pick the eclipse window

We use SPICE's `eclipse_timestamps_kepler` to find the primary and secondary eclipse contact times so PHOEBE only computes inside the windows of interest.

In [32]:
phoebe.multiprocessing_set_nprocs(1)

b = phoebe.default_binary()
b.flip_constraint('mass@primary', solve_for='sma')
#b.set_value_all('ntriangles@primary', 5000)
b.set_value('period@binary@component', period)
b.set_value('q@binary@component', q)
b.set_value('ecc@binary@component', ecc)
b.set_value('mass@primary@component', primary_mass)
b.set_value_all('incl@binary', inclination)

In [33]:
_, t1_p, _, _, t4_p, _, t1_s, _, _, t4_s = eclipse_timestamps_kepler(
    b.get_parameter('mass@primary@component').value,
    b.get_parameter('mass@secondary@component').value,
    b.get_parameter('period@binary@component').value * DAYS_TO_YR,
    b.get_parameter('ecc@binary@component').value,
    b.get_parameter('t0_perpass@binary@component').value * DAYS_TO_YR,
    jnp.deg2rad(b.get_parameter('incl@binary@component').value),
    b.get_parameter('per0@binary@component').value * DEG_TO_RAD,
    b.get_parameter('long_an@binary@component').value * DAYS_TO_YR,
    b.get_parameter('requiv@primary@component').value,
    b.get_parameter('requiv@secondary@component').value,
    pad=1.1,
    los_vector=jnp.array([0., 0., -1.]),
)

eclipse_edges = (float(t1_p), float(t4_p), float(t1_s), float(t4_s))
if not all(np.isfinite(eclipse_edges)):
    raise RuntimeError(f'No eclipse for this geometry: edges={eclipse_edges}')

half = int(n_times / 2)
times = np.concatenate([
    np.linspace(eclipse_edges[0] / DAYS_TO_YR, eclipse_edges[1] / DAYS_TO_YR, half),
    np.linspace(eclipse_edges[2] / DAYS_TO_YR, eclipse_edges[3] / DAYS_TO_YR, half),
])
times_yr = times * DAYS_TO_YR
print(f'Primary eclipse window  [d]: {eclipse_edges[0] / DAYS_TO_YR:.4f} - {eclipse_edges[1] / DAYS_TO_YR:.4f}')
print(f'Secondary eclipse window [d]: {eclipse_edges[2] / DAYS_TO_YR:.4f} - {eclipse_edges[3] / DAYS_TO_YR:.4f}')

Primary eclipse window  [d]: -0.1027 - 0.1027
Secondary eclipse window [d]: 2.3973 - 2.6027


In [ ]:
b.add_dataset('mesh', compute_times=times, columns=_mesh_columns, dataset='mesh01')
b.add_dataset('orb', compute_times=times, dataset='orb01')
b.add_dataset('lc', compute_times=times,
              passband='Bolometric:900-40000', dataset='lc_bolometric')
b.set_value_all('pblum_mode', dataset='lc_bolometric', value='absolute')
b.set_value('distance', 10 * u.pc)
b.set_value_all('ld_mode', 'manual')
b.set_value_all('ld_func', 'linear')
b.set_value_all('ld_coeffs', [0.])
b.set_value_all('ld_mode_bol', 'manual')
b.set_value_all('ld_func_bol', 'linear')
b.set_value_all('ld_coeffs_bol', [0.])
b.set_value_all('atm', 'blackbody')
b.set_value_all('irrad_method', 'none')
b.set_value_all('gravb_bol', 0.0)

b.compute_pblums(pbflux=True, set_value=True)
b.compute_ld_coeffs(ld_mode='manual', ld_func='linear', ld_coeffs=[0.])
b.run_compute(irrad_method='none', coordinates='uvw', ltte=False, ntriangles=n_mesh_elements)

fluxes_phoebe = np.asarray(b.get_parameter('fluxes@lc_bolometric@model').value)

100%|██████████| 40/40 [00:05<00:00,  7.67it/s]


In [35]:
len(b.get_value('us@primary', time=times[0]))

5746

## SPICE bolometric light curve

Same orbital parameters, blackbody atmosphere, no limb darkening — matched as closely as possible to the PHOEBE setup above.

In [ ]:
bb = Blackbody()
bol = Bolometric()

def default_icosphere(mass):
    return get_mesh_view(
        IcosphereModel.construct(n_mesh_elements, 1., mass,
                                 bb.solar_parameters, bb.parameter_names),
        jnp.array([0., 0., -1.]),
    )

body1 = default_icosphere(b.get_parameter('mass@primary@component').value)
body2 = default_icosphere(b.get_parameter('mass@secondary@component').value)
binary = Binary.from_bodies(body1, body2)
binary = add_orbit(
    binary,
    P=b.get_parameter('period@binary@component').value * DAYS_TO_YR,
    ecc=b.get_parameter('ecc@binary@component').value,
    T=b.get_parameter('t0_perpass@binary@component').value * DAYS_TO_YR,
    i=jnp.deg2rad(b.get_parameter('incl@binary@component').value),
    omega=b.get_parameter('per0@binary@component').value * DEG_TO_RAD,
    Omega=b.get_parameter('long_an@binary@component').value * DAYS_TO_YR,
    vgamma=b.get_parameter('vgamma').value,
    reference_time=b.get_parameter('t0_ref@binary@component').value * DAYS_TO_YR,
    mean_anomaly=b.get_parameter('mean_anom@binary@component').value * DEG_TO_RAD,
    orbit_resolution_points=len(times_yr),
)

pb1, pb2 = evaluate_orbit_at_times(binary, times_yr)

In [ ]:
vws = jnp.linspace(900, 40000, 1000)
log_vws = jnp.log10(vws)

bol_lum = []
for _pb1, _pb2 in zip(pb1, pb2):
    spec1 = simulate_observed_flux(bb.intensity, _pb1, log_vws, disable_doppler_shift=True)
    spec2 = simulate_observed_flux(bb.intensity, _pb2, log_vws, disable_doppler_shift=True)
    bol_lum.append(AB_passband_luminosity(bol, vws, spec1[:, 0] + spec2[:, 0]))
bol_lum = np.array(bol_lum)
spice_mag = bol_lum - bol_lum[0]

phoebe_mag_raw = -2.5 * np.log10(np.asarray(fluxes_phoebe, dtype=float))
phoebe_mag = phoebe_mag_raw - phoebe_mag_raw[0]
residual = spice_mag - phoebe_mag

## Plot the comparison

Top row: PHOEBE vs SPICE bolometric magnitudes (zeroed at $t_0$).
Bottom row: SPICE - PHOEBE residual.

In [ ]:
t_primary   = times[:half]
t_secondary = times[half:]

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex='col', sharey='row',
                         gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05, 'wspace': 0.15})

for ax_lc, ax_res, t, sl, title in (
    (axes[0, 0], axes[1, 0], t_primary,   slice(0, half),  'Primary eclipse'),
    (axes[0, 1], axes[1, 1], t_secondary, slice(half, None), 'Secondary eclipse'),
):
    ax_lc.plot(t, phoebe_mag[sl], 'o-', label='PHOEBE', color='black')
    ax_lc.plot(t, spice_mag[sl],  's--', label='SPICE',  color='firebrick')
    ax_lc.invert_yaxis()
    ax_lc.set_title(title)
    ax_lc.grid(alpha=0.3)

    ax_res.axhline(0, color='gray', lw=0.7)
    ax_res.plot(t, residual[sl], 'o-', color='steelblue')
    ax_res.set_xlabel('Time [days]')
    ax_res.grid(alpha=0.3)

axes[0, 0].set_ylabel(r'$\Delta$ mag (zeroed at $t_0$)')
axes[1, 0].set_ylabel('SPICE - PHOEBE [mag]')
axes[0, 0].legend(loc='best')
fig.suptitle(
    f'incl={inclination}, P={period} d, q={q}, ecc={ecc}, '
    f'm1={primary_mass}, n_mesh={n_mesh_elements}'
)
plt.show()

print(f'max |residual| = {np.nanmax(np.abs(residual)):.4e} mag')
print(f'rms residual   = {np.sqrt(np.nanmean(residual**2)):.4e} mag')

## (Optional) Save in the same pickle layout as the grid script

Useful if you want to drop the result into the existing `blackbody_comparison.ipynb` loader.

In [ ]:
import pickle
from pathlib import Path

SAVE = False  # flip to True to write the pickle
OUTPUT_PATH = Path('lc_eclipse')

if SAVE:
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    out_pkl = OUTPUT_PATH / (
        f'eclipses_incl_{inclination}_period_{period}_q_{q}_ecc_{ecc}'
        f'_primary_mass_{primary_mass}_nmesh_{n_mesh_elements}.pkl'
    )
    with open(out_pkl, 'wb') as f:
        pickle.dump({
            'phoebe_binary': b,
            'spice_body1': pb1,
            'spice_body2': pb2,
            'fluxes_phoebe': fluxes_phoebe,
            'bol_lum': spice_mag,
            'times': times,
            'n_times': n_times,
            'sma': b.get_parameter('sma@binary@component').value,
            'primary_mass': primary_mass,
            'secondary_mass': b.get_parameter('mass@secondary@component').value,
            'inclination': inclination,
            'period': period,
            'q': q,
            'ecc': ecc,
            'n_mesh_elements': n_mesh_elements,
        }, f)
    print(f'wrote {out_pkl}')